# Yelp Dataset — Google Drive → Extract → Convert to Parquet
**Steps:**
1. Mount Google Drive
2. Extract the ZIP/TAR file
3. Convert each JSON file to Parquet
4. Save Parquet files back to the same Google Drive folder

## Step 1 — Install Dependencies

In [1]:
!pip install pyarrow fastparquet -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.1 MB/s eta 0:00:00


## Step 2 — Import Libraries

In [2]:
import os
import tarfile
import zipfile
import shutil
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

## Step 3 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 4 — Configure Paths
> **Change `GDRIVE_FOLDER_PATH` to the exact folder path where your ZIP/TAR file is stored in Google Drive.**

In [4]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────────
GDRIVE_FOLDER_PATH = "/content/drive/MyDrive/Dataset"
ARCHIVE_FILENAME   = "Yelp-JSON.zip"
# ────────────────────────────────────────────────────────────────────────────

ARCHIVE_PATH  = os.path.join(GDRIVE_FOLDER_PATH, ARCHIVE_FILENAME)
EXTRACT_DIR   = "/content/yelp_extracted"   # temporary local extraction folder
PARQUET_DIR   = os.path.join(GDRIVE_FOLDER_PATH, "parquet")  # output folder in Drive

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR, exist_ok=True)

print(f"Archive  : {ARCHIVE_PATH}")
print(f"Extract  : {EXTRACT_DIR}")
print(f"Parquet  : {PARQUET_DIR}")
print(f"Archive exists: {os.path.exists(ARCHIVE_PATH)}")

Archive  : /content/drive/MyDrive/Dataset/Yelp-JSON.zip
Extract  : /content/yelp_extracted
Parquet  : /content/drive/MyDrive/Dataset/parquet
Archive exists: True


## Step 5 — Extract the Archive

In [5]:
def extract_archive(archive_path, extract_dir):
    print(f"Extracting {archive_path} ...")
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path, 'r') as z:
            z.extractall(path=extract_dir)
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path, 'r:*') as tar:
            tar.extractall(path=extract_dir, filter='data')
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")
    print("Extraction complete.")

extract_archive(ARCHIVE_PATH, EXTRACT_DIR)

# Handle nested archives — skip macOS metadata files (._xxx)
nested_archives = [
    f for f in Path(EXTRACT_DIR).rglob("*.tar")
    if not f.name.startswith("._")
] + [
    f for f in Path(EXTRACT_DIR).rglob("*.tgz")
    if not f.name.startswith("._")
]

for nested in nested_archives:
    print(f"\nFound nested archive: {nested.name} — extracting ...")
    with tarfile.open(nested, 'r:*') as tar:
        tar.extractall(path=EXTRACT_DIR, filter='data')
    print("  Nested extraction complete.")
    nested.unlink()

# Final JSON file list (skip macOS metadata)
extracted_files = [
    f for f in Path(EXTRACT_DIR).rglob("*.json")
    if not f.name.startswith("._")
]
print(f"\nFound {len(extracted_files)} JSON file(s):")
for f in extracted_files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"  {f.name}  ({size_mb:.1f} MB)")

Extracting /content/drive/MyDrive/Dataset/Yelp-JSON.zip ...
Extraction complete.

Found nested archive: yelp_dataset.tar — extracting ...
  Nested extraction complete.

Found 5 JSON file(s):
  yelp_academic_dataset_user.json  (3207.5 MB)
  yelp_academic_dataset_tip.json  (172.2 MB)
  yelp_academic_dataset_business.json  (113.4 MB)
  yelp_academic_dataset_checkin.json  (273.7 MB)
  yelp_academic_dataset_review.json  (5094.4 MB)


## Step 6 — Convert Each JSON File to Parquet
Large files (e.g. `review`, `user`) are processed in **chunks** to avoid running out of RAM.

In [6]:
# Files larger than this threshold are chunked (in MB)
CHUNK_THRESHOLD_MB = 500
CHUNK_SIZE         = 100_000  # rows per chunk


def json_to_parquet(json_path: Path, output_dir: str):
    output_path = os.path.join(output_dir, json_path.stem + ".parquet")
    file_size_mb = json_path.stat().st_size / (1024 ** 2)
    print(f"\nProcessing: {json_path.name}  ({file_size_mb:.1f} MB)")

    if file_size_mb > CHUNK_THRESHOLD_MB:
        # ── Chunked write for large files ──────────────────────────────────
        print(f"  Large file — using chunked mode ({CHUNK_SIZE:,} rows/chunk)")
        writer = None
        total_rows = 0
        for chunk in pd.read_json(json_path, lines=True, chunksize=CHUNK_SIZE):
            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema, compression='snappy')
            writer.write_table(table)
            total_rows += len(chunk)
        if writer:
            writer.close()
        print(f"  Done — {total_rows:,} rows written")
    else:
        # ── Single read for smaller files ───────────────────────────────────
        df = pd.read_json(json_path, lines=True)
        df.to_parquet(output_path, engine='pyarrow', compression='snappy', index=False)
        print(f"  Done — {len(df):,} rows written")

    out_size_mb = os.path.getsize(output_path) / (1024 ** 2)
    print(f"  Saved → {output_path}  ({out_size_mb:.1f} MB)")


for json_file in extracted_files:
    json_to_parquet(json_file, PARQUET_DIR)

print("\nAll files converted successfully!")


Processing: yelp_academic_dataset_user.json  (3207.5 MB)
  Large file — using chunked mode (100,000 rows/chunk)
  Done — 1,987,897 rows written
  Saved → /content/drive/MyDrive/Dataset/parquet/yelp_academic_dataset_user.parquet  (2501.7 MB)

Processing: yelp_academic_dataset_tip.json  (172.2 MB)
  Done — 908,915 rows written
  Saved → /content/drive/MyDrive/Dataset/parquet/yelp_academic_dataset_tip.parquet  (80.2 MB)

Processing: yelp_academic_dataset_business.json  (113.4 MB)
  Done — 150,346 rows written
  Saved → /content/drive/MyDrive/Dataset/parquet/yelp_academic_dataset_business.parquet  (17.5 MB)

Processing: yelp_academic_dataset_checkin.json  (273.7 MB)
  Done — 131,930 rows written
  Saved → /content/drive/MyDrive/Dataset/parquet/yelp_academic_dataset_checkin.parquet  (125.2 MB)

Processing: yelp_academic_dataset_review.json  (5094.4 MB)
  Large file — using chunked mode (100,000 rows/chunk)
  Done — 6,990,280 rows written
  Saved → /content/drive/MyDrive/Dataset/parquet/yel

## Step 7 — Verify Output

In [7]:
print("Parquet files saved in Google Drive:\n")
print(f"{'File':<45} {'Size (MB)':>10}  {'Rows':>12}")
print("-" * 70)

for pf in sorted(Path(PARQUET_DIR).glob("*.parquet")):
    size_mb = pf.stat().st_size / (1024 ** 2)
    df_meta = pq.read_metadata(str(pf))
    rows = df_meta.num_rows
    print(f"{pf.name:<45} {size_mb:>10.1f}  {rows:>12,}")

Parquet files saved in Google Drive:

File                                           Size (MB)          Rows
----------------------------------------------------------------------
yelp_academic_dataset_business.parquet              17.5       150,346
yelp_academic_dataset_checkin.parquet              125.2       131,930
yelp_academic_dataset_review.parquet              2859.6     6,990,280
yelp_academic_dataset_tip.parquet                   80.2       908,915
yelp_academic_dataset_user.parquet                2501.7     1,987,897


## Step 8 — Quick Schema Preview (Optional)

In [8]:
for pf in sorted(Path(PARQUET_DIR).glob("*.parquet")):
    print(f"\n── {pf.name} ──")
    df_sample = pd.read_parquet(pf, engine='pyarrow').head(2)
    print(df_sample.dtypes.to_string())
    display(df_sample)


── yelp_academic_dataset_business.parquet ──
business_id      object
name             object
address          object
city             object
state            object
postal_code      object
latitude        float64
longitude       float64
stars           float64
review_count      int64
is_open           int64
attributes       object
categories       object
hours            object


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,"{'AcceptsInsurance': None, 'AgesAllowed': None...","Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,"{'AcceptsInsurance': None, 'AgesAllowed': None...","Shipping Centers, Local Services, Notaries, Ma...","{'Friday': '8:0-18:30', 'Monday': '0:0-0:0', '..."



── yelp_academic_dataset_checkin.parquet ──
business_id    object
date           object


,business_id,date
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020..."
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011..."



── yelp_academic_dataset_review.parquet ──
review_id              object
user_id                object
business_id            object
stars                   int64
useful                  int64
funny                   int64
cool                    int64
text                   object
date           datetime64[ns]


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18



── yelp_academic_dataset_tip.parquet ──
user_id                     object
business_id                 object
text                        object
date                datetime64[ns]
compliment_count             int64


,user_id,business_id,text,date,compliment_count
0,AGNUgVwnZUey3gcPCJ76iw,3uLgwr0qeCNMjKenHJwPGQ,Avengers time with the ladies.,2012-05-18 02:17:21,0
1,NBN4MgHP9D3cw--SnauTkA,QoezRbYQncpRqyrLH6Iqjg,They have lots of good deserts and tasty cuban...,2013-02-05 18:35:10,0



── yelp_academic_dataset_user.parquet ──
user_id                object
name                   object
review_count            int64
yelping_since          object
useful                  int64
funny                   int64
cool                    int64
elite                  object
friends                object
fans                    int64
average_stars         float64
compliment_hot          int64
compliment_more         int64
compliment_profile      int64
compliment_cute         int64
compliment_list         int64
compliment_note         int64
compliment_plain        int64
compliment_cool         int64
compliment_funny        int64
compliment_writer       int64
compliment_photos       int64


,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946


## Step 9 — Cleanup Temp Files

In [9]:
shutil.rmtree(EXTRACT_DIR)
print(f"Deleted temporary folder: {EXTRACT_DIR}")

Deleted temporary folder: /content/yelp_extracted
